# 04 - Agent Build & Test

**Pre-req**: Notebooks 00–03 complete.

In [1]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# Confirm key env vars loaded
assert os.getenv('GROQ_API_KEY'), 'GROQ_API_KEY missing from .env'
print(f'GROQ key loaded: {os.getenv("GROQ_API_KEY")[:8]}...')

# agent.py and tools.py are in project root
sys.path.insert(0, str(Path('..').resolve()))

from agent import create_agent, run_agent, run_agent_stream
print('Imports OK')

GROQ key loaded: gsk_vKiL...
Imports OK


## 1. Build agent

In [2]:
agent = create_agent()
print('Agent ready')
print("Tools: ['weaviate_search', 'neo4j_graph_search']")

Agent ready
Tools: ['weaviate_search', 'neo4j_graph_search']


/Users/abhirup/Developments/scm-graph-rag/agent.py:55: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  return create_react_agent(


## 2. Single turn

In [3]:
response = run_agent(
    agent,
    query='What are the main supply chain risks for electronics manufacturers?',
    thread_id='test-thread-001',
)
print(response)

According to [1], the main supply chain risks for electronics manufacturers include disruptions in transportation, which can be mitigated by implementing Bayes adaptive Monte Carlo tree search for offline model-based reinforcement learning. Additionally, [31] suggests that a modified failure mode and effects analysis method can be used for supplier selection problems in the supply chain risk environment.

Graph: Manufacturing --[PART_OF]--> Distribution, Manufacturing --[PART_OF]--> supply chains, and Risks --[AFFECTS]--> Businesses, indicate that manufacturing is a critical component of the supply chain and is affected by various risks.

As stated in [32], it is essential to acknowledge the risks related to the location of an external supplier, including political stability, safety, and lead times of shipment schedules. Moreover, [supply_chain_management_tutorial.pdf] highlights the importance of examining potential secondary carriers or routes and searching for other producers as a b

## 3. Multi-turn (memory via PostgresSaver)

In [4]:
THREAD = 'test-multiturn-001'

r1 = run_agent(agent, 'What is just-in-time inventory?', THREAD)
print('Turn 1:', r1[:300])

r2 = run_agent(agent, 'How does it affect supplier relationships?', THREAD)
print('\nTurn 2 (uses context from turn 1):')
print(r2[:300])

Turn 1: According to [supply_chain_management_tutorial.pdf], just-in-time inventory is a management approach that aims to maintain minimal inventory levels by producing and delivering products just in time to meet customer demand. This approach helps companies minimize waste, reduce costs, and improve effic

Turn 2 (uses context from turn 1):
According to [supply_chain_management_tutorial.pdf], just-in-time inventory management can significantly impact supplier relationships. By maintaining minimal inventory levels, companies can reduce their reliance on suppliers and improve their negotiating power. 

Graph: Supplier 2 --[SUPPLIES_TO]--


## 4. Streaming

In [6]:
print('Streaming:')
print('-' * 60)
full = ''
for event in run_agent_stream(agent, 'Which suppliers are at risk from port congestion?', 'test-stream-001'):
    if event["type"] == "token":
        print(event["content"], end='', flush=True)
        full += event["content"]
    elif event["type"] == "tool_call":
        print(f"\n[Tool] {event['tool']}: {event.get('query','')}", flush=True)
    elif event["type"] == "tool_result":
        print(f"\n[Done] {event['tool']}", flush=True)
print(f'\nTotal chars: {len(full)}')

Streaming:
------------------------------------------------------------

[Tool] neo4j_graph_search: port congestion

[Tool] weaviate_search: port congestion supply chain risk suppliers

[Done] neo4j_graph_search

[Done] weaviate_search
According to [1], port congestion can trigger widespread instability across the entire supply chain system. Graph: Transportation --[SHIPS_TO]--> Customer, and Graph: Ports --[LOCATED_IN]--> Transportation, indicate that suppliers who ship to customers through ports are at risk from port congestion. 

As stated in [2], traditional supply chain models often fail to account for intricate interdependencies across multi-level networks, making them vulnerable to disruptions such as port congestion. 

[3] suggests that proactive planning, risk management, and real-time optimization can help mitigate the effects of port congestion. 

[4] highlights the importance of examining the safety and lead times of shipment schedules, as well as labeling and examining pot

## 5. Verify checkpoints

In [7]:
import os, psycopg
from urllib.parse import quote

DB_URI = (
    f'postgresql://{quote(os.getenv("PG_USER"), safe="")}:{quote(os.getenv("PG_PASSWORD"), safe="")}'
    f'@{os.getenv("PG_HOST","localhost")}:{os.getenv("PG_PORT","5432")}/{os.getenv("PG_DATABASE","postgres")}'
)

with psycopg.connect(DB_URI) as conn:
    rows = conn.execute(
        'SELECT thread_id, COUNT(*) AS c FROM checkpoints GROUP BY thread_id ORDER BY c DESC'
    ).fetchall()
    print('Checkpoints per thread:')
    for thread_id, count in rows:
        print(f'  {thread_id}: {count}')

Checkpoints per thread:
  test-multiturn-001: 10
  test-stream-001: 7
  test-thread-001: 5
